# model_inference — best model -> Kaggle submission (Weights & Biases)

1. Query the wandb `final` runs across all architecture groups.
2. Pick the lowest `holdout_wmae` and **promote its model artifact** with the
   `best` alias (this is the wandb model-registry entry).
3. **Load it back from wandb** (`walmart_<arch>:best`) and predict on the RAW test set.
4. Write `submissions/submission.csv` in Kaggle format (`Id,Weekly_Sales`).

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
os.environ.setdefault("WANDB_SILENT", "true")
import numpy as np, pandas as pd
import wandb

from src.data import load_raw, make_submission_id
from src.pipeline import RAW_COLS
from src.wandb_utils import (WANDB_PROJECT, WANDB_ENTITY, load_pipeline, project_path)

raw = load_raw("data")
test = raw.test
api = wandb.Api()
path = project_path(api)
print("querying wandb runs at:", path)

## 1. Find the best `final` run across architectures

In [ ]:
rows, best_run, best_wmae = [], None, float("inf")
for r in api.runs(path):
    if r.job_type != "final":
        continue
    w = r.summary.get("holdout_wmae")
    if w is None:
        continue
    rows.append((r.group, r.name, round(float(w), 2)))
    if w < best_wmae:
        best_wmae, best_run = float(w), r

print(pd.DataFrame(rows, columns=["group", "run", "holdout_wmae"])
      .sort_values("holdout_wmae").to_string(index=False))
print("\nBEST:", best_run.name, "| WMAE =", round(best_wmae, 2))

## 2. Promote the best model artifact -> `best` alias (registry)

In [ ]:
model_art = next(a for a in best_run.logged_artifacts() if a.type == "model")
collection = model_art.name.split(":")[0]          # e.g. 'walmart_lightgbm'
if "best" not in model_art.aliases:
    model_art.aliases.append("best")
    model_art.save()
print("registered:", collection, "| version:", model_art.version, "| aliases:", model_art.aliases)

## 3. Load from wandb and predict on RAW test

In [ ]:
run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY,
                 job_type="inference", name="inference")
used = run.use_artifact(f"{collection}:best")
model = load_pipeline(used.download())
preds = np.clip(model.predict(test[RAW_COLS]), 0, None)   # raw test; sales >= 0
run.log({"n_predictions": len(preds), "pred_mean": float(preds.mean())})
print("predicted", len(preds), "rows | mean =", round(float(preds.mean()), 2))

## 4. Build the Kaggle submission

In [ ]:
os.makedirs("submissions", exist_ok=True)
submission = pd.DataFrame({"Id": make_submission_id(test), "Weekly_Sales": preds})
submission.to_csv("submissions/submission.csv", index=False)
sub_art = wandb.Artifact("submission", type="submission")
sub_art.add_file("submissions/submission.csv")
run.log_artifact(sub_art)
run.finish()
print(submission.shape); submission.head()

### Upload to Kaggle (optional)
Needs `~/.kaggle/kaggle.json` credentials.
```bash
kaggle competitions submit -c walmart-recruiting-store-sales-forecasting \
  -f submissions/submission.csv -m "best model via wandb registry"
```